# Do raw coordinates or simple features survive a reference-frame change? An analysis of the HPM gesture dataset

**Author:** Yashi Saxena

**Dataset:** Verduyn, A. (2025). *HPM dataset: a dataset of recorded Hand Palm Motion gestures.* Zenodo. https://doi.org/10.5281/zenodo.15020058 (CC BY 4.0). The data are not included in this repository; download them from Zenodo.

**Scope:** HPM is a gesture-recognition dataset, not a motor-learning dataset. This notebook describes and compares movement trajectories. It makes no claims about learning or practice effects.

## Summary of findings

1. **Raw coordinates are not comparable across reference frames, but the difference is a rigid transformation.** Between `Original_2` and `displaced_world_1`, the raw trajectories differ by an average RMS of 1.36 (position units). After a best-fit rotation and translation (Kabsch alignment) the residual is 0.0014, a 99.9% reduction, in all 35 matched recordings. The estimated rotation is about 180° around the Z axis, matching the documented transformation.
2. **Feature stability depends on the transformation.** Across the three world-frame contexts, mean path length changed by less than 0.06% and straightness by less than 1%. Across the three body-frame contexts, path length rose by 1.3% to 7.3% and straightness changed by −10.2% to +3.3%, because the tracked origin moves with the rotating hand. Total rotation changed by less than 1.2% in all six contexts.
3. **Summed features are sensitive to sampling and noise.** Mean path length and total rotation rise with the number of samples per recording (150, 300, 598 samples: path length 1.05, 1.18, 1.46), and `Original_2` has about 0.2 more path length than `Original_1` for every gesture. This is consistent with small point-to-point noise accumulating in the sums. I did not test this directly.
4. **Trial-to-trial variability differs by gesture and by how it is measured.** Path-length variability (coefficient of variation) ranged from 2.0% (`OpenGripper`) to 10.8% (`GoHome`); rotation variability from 0.5% (`CloseGripper`) to 6.0% (`GoRight`). `GoHome` deviates more from its mean trajectory than `GoDown` in absolute terms (2.3 times), but less relative to its movement size.
5. **PCA:** four engineered features concentrate 81% of their variance in two components, against 48% for 350 raw values. This describes compactness only, not gesture separability.
6. **Classification under context change:** raw coordinates classify almost perfectly when the frame is unchanged but fall to chance (0.14) in all three world-frame contexts. Simple features trained on `Original_2` reach 0.57 to 0.64 in the world-frame contexts, but are worse than raw coordinates when the frame is unchanged and fail on truncated trajectories. Results depend strongly on the training context. Each context has only 14 test samples, so these are exploratory results.

## 1. Setup and data loading

The dataset has 7 gestures (`CloseGripper`, `GoDown`, `GoHome`, `GoLeft`, `GoRight`, `GoUp`, `OpenGripper`). Each gesture has 12 *contexts*, and each context has 5 trials, giving 7 × 12 × 5 = 420 recordings. The contexts, according to the dataset description, are:

| Context | Description |
|---|---|
| `Original_1` | The original recordings |
| `Original_2` | A baseline with no artificial transformation |
| `Slower`, `Faster` | Time axis rescaled and resampled (about half and twice the speed) |
| `First_Half` | Only the first half of each trajectory (an unfinished gesture) |
| `displaced_body_1-3` | Body frame shifted 5 cm and rotated |
| `displaced_world_1-3` | World frame shifted 1 m and rotated |
| `Combination` | Faster, first half only, and both frames changed |

Each file contains time, x, y, z position and a four-value quaternion (scalar first) at nominal 0.02 s intervals. The original recordings were 5 trials per gesture (35 recordings); the other contexts were created by transforming or perturbing them.

**To run:** set `ROOT` to the folder containing the seven gesture folders. In Google Colab, upload the unzipped dataset to Google Drive first.

In [ ]:
# Google Colab only: mount Google Drive. Skipped automatically in other environments.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Set this to the folder containing CloseGripper, GoDown, GoHome, ...
ROOT = Path("/content/drive/MyDrive/HPM/HPM")

GESTURES = ["CloseGripper", "GoDown", "GoHome", "GoLeft", "GoRight", "GoUp", "OpenGripper"]

In [ ]:
def load_trial(path):
    # Columns: time, x, y, z, then 4 quaternion values (scalar first).
    df = pd.read_csv(path, header=None)
    df = df.apply(pd.to_numeric, errors="coerce")  # a header row, if any, becomes NaN
    df = df.dropna(how="any")
    return df.iloc[:, :8].to_numpy(dtype=float)


def trial_number(path, fallback):
    m = re.search(r"(\d+)", path.stem)
    return int(m.group(1)) if m else fallback


records = []
for gesture_dir in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    for context_dir in sorted(p for p in gesture_dir.iterdir() if p.is_dir()):
        for i, f in enumerate(sorted(context_dir.glob("*.csv"))):
            records.append(dict(gesture=gesture_dir.name, context=context_dir.name,
                                trial=trial_number(f, i + 1), data=load_trial(f)))

meta = pd.DataFrame([{k: v for k, v in r.items() if k != "data"} for r in records])
index = {(r["gesture"], r["context"], r["trial"]): r["data"] for r in records}


def get(gesture, context, trial):
    return index[(gesture, context, trial)]


print(len(records), "trials loaded")
print("Unique gestures:", meta["gesture"].nunique(), "| unique contexts:", meta["context"].nunique())
print("Trials per gesture-context:", meta.groupby(["gesture", "context"]).size().unique().tolist())
print("Duplicate gesture-context-trial keys:", int(meta.duplicated(["gesture", "context", "trial"]).sum()))

### Data integrity check

Before analysis, I checked that every file has the expected columns, no missing values, and consistent sampling, and how long the recordings are in each context.

In [ ]:
audit = pd.DataFrame([{
    "context": r["context"],
    "rows": len(r["data"]),
    "duration_s": r["data"][-1, 0] - r["data"][0, 0],
    "median_dt": float(np.median(np.diff(r["data"][:, 0]))),
    "has_missing": bool(np.isnan(r["data"]).any()),
} for r in records])

print("Files with missing values:", int(audit["has_missing"].sum()))
print("Median sampling interval(s):", sorted(audit["median_dt"].round(3).unique().tolist()))

context_lengths = audit.groupby("context").agg(
    samples_per_recording=("rows", "unique"), mean_duration_s=("duration_s", "mean")
)
context_lengths["samples_per_recording"] = context_lengths["samples_per_recording"].apply(list)
display(context_lengths.round(2))

**Interpretation.** All 420 recordings loaded without missing values and with a regular 0.02 s sampling interval. The number of samples depends on the context: 300 for `Original_1`, `Original_2` and the six displaced contexts (about 6 s), 598 for `Slower` (about 12 s), 150 for `Faster` and `First_Half` (about 3 s) and 75 for `Combination` (about 1.5 s). This matters later, because features that are summed over samples depend on how many samples a recording has.

## 2. The reference-frame problem

The same gesture, `GoRight` (trial 1), is shown below in three contexts: the original recording, the baseline `Original_2`, and `displaced_world_1`.

In [ ]:
gesture = "GoRight"
first_trial = int(meta["trial"].min())
show_contexts = ["Original_1", "Original_2", "displaced_world_1"]

fig = plt.figure(figsize=(16, 5))
for k, ctx in enumerate(show_contexts):
    d = get(gesture, ctx, first_trial)
    ax = fig.add_subplot(1, 3, k + 1, projection="3d")
    ax.plot(d[:, 1], d[:, 2], d[:, 3])
    ax.set_title(f"{gesture}: {ctx}")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
plt.tight_layout()
plt.show()


def path_length(xyz):
    return np.linalg.norm(np.diff(xyz, axis=0), axis=1).sum()


pl = {ctx: path_length(get(gesture, ctx, first_trial)[:, 1:4]) for ctx in show_contexts}
for ctx, v in pl.items():
    print(f"{ctx:18s} path length: {v:.4f}")
print(f"displaced_world_1 vs Original_2: {100 * (pl['displaced_world_1'] - pl['Original_2']) / pl['Original_2']:+.2f}%")
print(f"displaced_world_1 vs Original_1: {100 * (pl['displaced_world_1'] - pl['Original_1']) / pl['Original_1']:+.2f}%")

**Observation.** `Original_1` and `Original_2` have similar trajectories, while `displaced_world_1` occupies a completely different coordinate region and looks differently oriented. (The 3D panels autoscale their axes, so shapes look different partly because of the viewing angle; the quantitative check below does not rely on how they look.)

The path length of `displaced_world_1` (1.3112) is within 0.5% of `Original_2` (1.3178) but about 20% above `Original_1` (1.0917). So the displaced recording appears to be derived from `Original_2`, not directly from `Original_1`.

**Interpretation.** A gesture can be described by very different coordinates depending on the reference frame, so raw coordinates cannot be compared directly across contexts. The next section tests whether the difference is only a rigid rotation and translation.

## 3. Is a world-frame change a rigid transformation?

If the world frame is simply moved and rotated, the displaced trajectory should be a rotated and shifted copy of the original. I tested this with the **Kabsch algorithm**, which finds the rotation and translation that best align two matched sets of 3D points, and measured the root-mean-square (RMS) distance between corresponding points before and after alignment.

I first examined one pair (`GoRight`, trial 1: `Original_2` vs `displaced_world_1`), then all 35 pairs (7 gestures × 5 trials).

In [ ]:
def rigid_align(A, B):
    # Rotate and translate A to best match B (Kabsch). Returns aligned A, rotation, RMS before, RMS after.
    A_mean, B_mean = A.mean(axis=0), B.mean(axis=0)
    Ac, Bc = A - A_mean, B - B_mean
    U, S, Vt = np.linalg.svd(Ac.T @ Bc)
    R = U @ Vt
    if np.linalg.det(R) < 0:        # prevent a reflection
        Vt[-1, :] *= -1
        R = U @ Vt
    A_aligned = Ac @ R + B_mean
    rms_before = np.sqrt(np.mean(np.sum((A - B) ** 2, axis=1)))
    rms_after = np.sqrt(np.mean(np.sum((A_aligned - B) ** 2, axis=1)))
    return A_aligned, R, rms_before, rms_after


d_orig = get("GoRight", "Original_2", first_trial)
d_disp = get("GoRight", "displaced_world_1", first_trial)
print("Same shape:", d_orig.shape == d_disp.shape, "| identical timestamps:", np.allclose(d_orig[:, 0], d_disp[:, 0]))

A, B = d_orig[:, 1:4], d_disp[:, 1:4]
A_aligned, R, rms_before, rms_after = rigid_align(A, B)

print("RMS before alignment:", round(rms_before, 5))
print("RMS after alignment: ", round(rms_after, 5))
print("Determinant of rotation:", round(float(np.linalg.det(R)), 6))
print("Estimated rotation matrix (rounded):")
print(np.round(R, 3))

seg_diff = np.abs(np.linalg.norm(np.diff(A, axis=0), axis=1) - np.linalg.norm(np.diff(B, axis=0), axis=1))
print("Segment-length difference: mean", round(seg_diff.mean(), 5), "| max", round(seg_diff.max(), 5))

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.plot(A[:, 0], A[:, 1], A[:, 2], label="Original_2")
ax.plot(B[:, 0], B[:, 1], B[:, 2], label="displaced_world_1")
ax.plot(A_aligned[:, 0], A_aligned[:, 1], A_aligned[:, 2], "--", label="Original_2 after alignment")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Rigid alignment of GoRight (trial 1)")
ax.legend()
plt.tight_layout()
plt.show()

**Result for one pair.** The RMS error fell from 1.52447 to 0.00138 (a 99.9% reduction). The rotation has a determinant of 1 (a proper rotation, not a reflection) and is approximately diag(−1, −1, 1), a 180° rotation about the Z axis. This is what the dataset documentation describes for `displaced_world_1` (1 m shift and a 180° rotation about Z). In the plot, the aligned original almost completely overlaps the displaced trajectory.

In [ ]:
rows = []
for g in GESTURES:
    for t in sorted(meta["trial"].unique()):
        o, dsp = get(g, "Original_2", t), get(g, "displaced_world_1", t)
        if o.shape != dsp.shape or not np.allclose(o[:, 0], dsp[:, 0]):
            continue                                   # only compare recordings with identical timestamps
        Ao, Bd = o[:, 1:4], dsp[:, 1:4]
        _, Rt, rb, ra = rigid_align(Ao, Bd)
        rows.append(dict(Gesture=g, Trial=t, Original_Path=path_length(Ao), Displaced_Path=path_length(Bd),
                         RMS_Before=rb, RMS_After=ra, Rotation_Determinant=np.linalg.det(Rt)))

results_35 = pd.DataFrame(rows)
results_35["Path_Difference_%"] = 100 * (results_35["Original_Path"] - results_35["Displaced_Path"]).abs() / results_35["Original_Path"]

print("Total comparisons:", len(results_35))
print("Mean RMS before alignment:", round(results_35["RMS_Before"].mean(), 4))
print("Mean RMS after alignment: ", round(results_35["RMS_After"].mean(), 6))
print("Range of RMS after alignment:", round(results_35["RMS_After"].min(), 5), "to", round(results_35["RMS_After"].max(), 5))
print("Mean path-length difference (%):", round(results_35["Path_Difference_%"].mean(), 2))
print("Maximum path-length difference (%):", round(results_35["Path_Difference_%"].max(), 2))
print("Mean rotation determinant:", round(results_35["Rotation_Determinant"].mean(), 6))

summary_35 = results_35.groupby("Gesture").agg(
    trials=("Trial", "count"),
    mean_RMS_before=("RMS_Before", "mean"),
    mean_RMS_after=("RMS_After", "mean"),
    mean_path_diff_pct=("Path_Difference_%", "mean"),
).round(5)
display(summary_35)

**Result for all 35 pairs.**

| Measure | Value |
|---|---:|
| Matched recording pairs | 35 (all had identical timestamps and shapes) |
| Mean RMS before alignment | 1.363 |
| Mean RMS after alignment | 0.0014 (range 0.00135 to 0.00148) |
| Reduction | about 99.9% |
| Mean path-length difference | 1.15% (largest single pair about 5.9%, `GoDown` trial 5) |
| Rotation determinants | 1 for every pair |

**Interpretation.**
- The large differences in raw coordinates are explained almost entirely by a rotation and translation, in every gesture and trial examined. If positions are in metres (the dataset describes the world-frame shift as 1 m), the remaining residual is about 1.4 mm.
- The residual is small but not zero, and it is similar for every pair. That is consistent with small independent perturbations in each file, not with an imperfect transformation. The dataset page mentions steps to prevent some contexts from exactly matching the others; I did not verify the exact procedure.
- Path length is nearly preserved (1.15% on average) but not exactly. The largest differences occur for `GoDown`, the shortest gesture, where small point-to-point noise makes up a larger share of the path. Section 4 returns to this.
- **Trial correspondence:** for `Original_2` and `displaced_world_1`, trial *k* matches trial *k* in all 35 cases (identical timestamps and near-zero residual after alignment). For the other contexts I assume the same correspondence, but I did not verify it individually.

**Limitation.** This tests one context pair (`Original_2` vs `displaced_world_1`) and position only (not orientation).

## 4. Simple features and their stability across contexts

If rotation and translation of the world frame do not change the geometry, features based on distances and angles should stay stable. I extracted six features per recording:

- **Path length:** the sum of distances between consecutive positions.
- **Total rotation:** the sum of angles between consecutive orientations (from the quaternions).
- **Straightness:** the distance between start and end divided by path length.
- **Net rotation:** the angle between the first and last orientation.
- **Duration** and **mean speed** (kept for description, but not used later for classification because they change with the Faster and Slower contexts).

In [ ]:
def quat_unit(q):
    norms = np.linalg.norm(q, axis=1, keepdims=True)
    if np.any(norms < 1e-12):
        raise ValueError("Zero-norm quaternion found; inspect the source trial.")
    return q / norms


def features(arr):
    t, pos, q = arr[:, 0], arr[:, 1:4], quat_unit(arr[:, 4:8])
    step = np.linalg.norm(np.diff(pos, axis=0), axis=1)
    path_len = step.sum()
    duration = t[-1] - t[0]
    # Angle between successive orientations; the absolute dot product handles q and -q describing the same rotation.
    dots = np.abs(np.sum(q[:-1] * q[1:], axis=1)).clip(0, 1)
    total_rotation = (2 * np.arccos(dots)).sum()
    net_disp = np.linalg.norm(pos[-1] - pos[0])
    net_rotation = 2 * np.arccos(np.abs(np.sum(q[0] * q[-1])).clip(0, 1))
    return dict(path_length=path_len, total_rotation=total_rotation,
                straightness=net_disp / path_len if path_len > 0 else 0.0,
                net_rotation=net_rotation, duration=duration,
                mean_speed=path_len / duration if duration > 0 else 0.0)


feat = pd.DataFrame([features(r["data"]) for r in records])
table = pd.concat([meta, feat], axis=1)
table["n_samples"] = [len(r["data"]) for r in records]

context_summary = table.groupby("context").agg(
    samples=("n_samples", "first"), duration=("duration", "mean"),
    path_length=("path_length", "mean"), total_rotation=("total_rotation", "mean")).round(3)
display(context_summary)

### 4a. World-frame changes

In [ ]:
FEATS3 = ["path_length", "total_rotation", "straightness"]
world_contexts = ["Original_2", "displaced_world_1", "displaced_world_2", "displaced_world_3"]
body_contexts = ["Original_2", "displaced_body_1", "displaced_body_2", "displaced_body_3"]


def compare_to_baseline(contexts):
    sub = table[table["context"].isin(contexts)]
    stats = sub.groupby("context")[FEATS3].agg(["mean", "std"]).loc[contexts]
    means = sub.groupby("context")[FEATS3].mean().loc[contexts]
    change = 100 * (means - means.loc["Original_2"]) / means.loc["Original_2"].abs()
    return stats, change


world_stats, world_change = compare_to_baseline(world_contexts)
display(world_stats.round(4))
print("Percentage change relative to Original_2:")
display(world_change.round(3))

**Result.** Compared with `Original_2`, the three world-frame contexts changed mean path length by +0.0006%, +0.054% and −0.030%; straightness by +0.68%, −0.83% and −0.34%; and total rotation by +0.33%, +0.84% and +1.13%.

**Interpretation.** Path length and straightness are essentially unchanged by a world-frame shift and rotation, as expected for a rigid transformation. Total rotation moves slightly more (up to 1.1%), though still very little. These are comparisons of context means, and the standard deviations (for example 0.44 for path length) show large differences between gestures, so this says the features are stable across contexts, not that they separate gestures.

### 4b. Body-frame changes

In [ ]:
body_stats, body_change = compare_to_baseline(body_contexts)
display(body_stats.round(4))
print("Percentage change relative to Original_2:")
display(body_change.round(3))

**Result.** Compared with `Original_2`, the three body-frame contexts changed mean path length by +1.35%, +2.94% and +7.30%; straightness by −8.81%, +3.26% and −10.20%; and total rotation by +0.52%, +0.80% and +0.14%.

**Interpretation.** Unlike a world-frame change, a body-frame change moves the point that is being tracked. The body frame is shifted by 5 cm and rotated, so when the hand rotates, the tracked origin swings along an additional path, which can alter both path length and straightness. Total rotation, which depends on orientation changes only, stayed stable. So a feature that is invariant to a global rigid transformation is not necessarily invariant to a change in the tracked point. I inferred this from the transformation description and the pattern of results; I did not isolate it experimentally.

### 4c. Speed, truncation, and sampling

Path length and total rotation are *sums over samples*. The contexts differ in the number of samples, so I checked how these sums relate to it.

In [ ]:
full_gesture_contexts = ["Faster", "Original_1", "Original_2", "Slower",
                         "displaced_body_1", "displaced_body_2", "displaced_body_3",
                         "displaced_world_1", "displaced_world_2", "displaced_world_3"]
cs = context_summary.loc[full_gesture_contexts]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, col, label in zip(axes, ["path_length", "total_rotation"], ["Mean path length", "Mean total rotation (rad)"]):
    ax.scatter(cs["samples"], cs[col])
    for name, row in cs.iterrows():
        if name in ("Faster", "Original_1", "Original_2", "Slower"):
            ax.annotate(name, (row["samples"], row[col]), textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.set_xlabel("Samples per recording")
    ax.set_ylabel(label)
plt.suptitle("Summed features vs number of samples (contexts containing the full gesture)")
plt.tight_layout()
plt.show()

per_gesture = table[table["context"].isin(["Original_1", "Original_2"])].pivot_table(
    index="gesture", columns="context", values="path_length", aggfunc="mean")
per_gesture["difference"] = per_gesture["Original_2"] - per_gesture["Original_1"]
display(per_gesture.round(3))

d1 = get("GoRight", "Original_1", first_trial)[:, 1:4]
d2 = get("GoRight", "Original_2", first_trial)[:, 1:4]
print("GoRight trial 1 coordinate ranges (min, max):")
print("  Original_1:", np.round(d1.min(axis=0), 4), np.round(d1.max(axis=0), 4))
print("  Original_2:", np.round(d2.min(axis=0), 4), np.round(d2.max(axis=0), 4))

**Result.**
- Among contexts containing the full gesture, mean path length rises with the number of samples: 1.045 for `Faster` (150 samples), 1.176 for `Original_2` (300) and 1.455 for `Slower` (598). Total rotation shows the same pattern (5.62, 7.74 and 12.43).
- `Original_2` has a higher mean path length than `Original_1` for all seven gestures, by about 0.2 in each case (range about 0.17 to 0.27), even though the coordinate ranges of the two contexts are almost identical (for `GoRight` trial 1, x from −0.135 to 0.117 in both).
- `Original_2` and all displaced contexts have a total rotation of about 7.7, against 4.0 for `Original_1`.

**Interpretation.** The underlying movement should be the same in all of these recordings, yet the sums depend on how many samples there are and which version of the data is used. A plausible explanation is small point-to-point noise: each step adds a little spurious distance and angle, so recordings with more steps, or with more noise per step, have larger sums. An almost constant additive difference between `Original_1` and `Original_2`, regardless of gesture size, fits this better than a genuine change in the movement. This is an explanation supported by several observations, not a tested conclusion. A direct test would smooth the data before computing the features and see whether the differences shrink.

**Consequence for later analysis.** Summed features such as path length and total rotation are not safe invariants across contexts that differ in sampling or noise, and the truncated contexts (`First_Half`, `Combination`) change them for a different reason: they contain only part of the gesture.

## 5. Trial-to-trial variability (original recordings)

Using `Original_1` (the original recordings), I asked how consistent the five repeated trials of each gesture are. Variability is shown as the **coefficient of variation (CV)**: the standard deviation divided by the mean, as a percentage, so gestures of different sizes can be compared.

In [ ]:
orig = table[table["context"] == "Original_1"]
variability = orig.groupby("gesture")[["path_length", "total_rotation", "duration"]].agg(["mean", "std"])
display(variability.round(3))

cv = pd.DataFrame({
    "path_length_cv": 100 * variability[("path_length", "std")] / variability[("path_length", "mean")],
    "rotation_cv": 100 * variability[("total_rotation", "std")] / variability[("total_rotation", "mean")],
})
display(cv.round(2))

ax = cv.plot(kind="bar", figsize=(11, 5), width=0.75)
ax.set_title("Trial-to-trial variability in movement features (Original_1)")
ax.set_xlabel("Gesture")
ax.set_ylabel("Coefficient of variation (%)")
plt.tight_layout()
plt.show()

**Result.**

| Gesture | Path-length CV (%) | Rotation CV (%) |
|---|---:|---:|
| CloseGripper | 9.12 | 0.53 |
| GoDown | 3.63 | 1.94 |
| GoHome | 10.75 | 5.19 |
| GoLeft | 6.79 | 5.95 |
| GoRight | 3.35 | 6.02 |
| GoUp | 3.70 | 3.31 |
| OpenGripper | 2.04 | 1.09 |

- Path-length variability is highest for `GoHome` (10.75%) and `CloseGripper` (9.12%), and lowest for `OpenGripper` (2.04%).
- Rotation variability is highest for `GoRight` (6.02%) and `GoLeft` (5.95%), and lowest for `CloseGripper` (0.53%).
- The two measures do not go together: `CloseGripper` is variable in path length but very consistent in rotation, and `GoRight` is the opposite. (`Duration` is 5.98 s for every recording, so it carries no information about variability.)

**Interpretation.** Consistency is feature-dependent, not a general property of a gesture. A performer can repeat how far the hand moves without repeating how it turns, or the reverse. With five trials per gesture, these are descriptive values, not statistically tested differences between gestures.

### Trajectory-level variability: `GoHome` vs `GoDown`

To look beyond summary features, the five `Original_1` trajectories of `GoHome` and `GoDown` were plotted, and each trial's RMS deviation from the gesture's mean trajectory was computed. This deviation is measured in the original coordinate frame, so it reflects both shape and position differences.

In [ ]:
fig = plt.figure(figsize=(14, 6))
for i, g in enumerate(["GoHome", "GoDown"]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    for t in [1, 2, 3, 4, 5]:
        d = get(g, "Original_1", t)
        ax.plot(d[:, 1], d[:, 2], d[:, 3], label=f"Trial {t}")
    ax.set_title(f"{g}: five repeated trials")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for g in ["GoHome", "GoDown"]:
    trajs = [get(g, "Original_1", t) for t in [1, 2, 3, 4, 5]]
    assert all(d.shape == trajs[0].shape and np.allclose(d[:, 0], trajs[0][:, 0]) for d in trajs), g
    xyz = np.stack([d[:, 1:4] for d in trajs])
    mean_traj = xyz.mean(axis=0)
    for k in range(5):
        rows.append(dict(Gesture=g, Trial=k + 1,
                         RMS_Deviation=np.sqrt(np.mean(np.sum((xyz[k] - mean_traj) ** 2, axis=1)))))

trajectory_df = pd.DataFrame(rows)
display(trajectory_df.round(5))

trajectory_summary = trajectory_df.groupby("Gesture")["RMS_Deviation"].agg(["mean", "std"])
mean_path = variability[("path_length", "mean")].loc[trajectory_summary.index]
trajectory_summary["mean_path_length"] = mean_path
trajectory_summary["deviation_per_path_length"] = trajectory_summary["mean"] / mean_path
display(trajectory_summary.round(5))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(trajectory_summary.index, trajectory_summary["mean"], yerr=trajectory_summary["std"], capsize=6)
axes[0].set_title("Absolute RMS deviation")
axes[0].set_ylabel("Mean RMS deviation from mean trajectory")
axes[1].bar(trajectory_summary.index, trajectory_summary["deviation_per_path_length"])
axes[1].set_title("RMS deviation relative to path length")
axes[1].set_ylabel("RMS deviation / mean path length")
plt.tight_layout()
plt.show()

**Result.**

| Gesture | Mean RMS deviation (SD) | Mean path length | Deviation ÷ path length |
|---|---:|---:|---:|
| GoDown | 0.0231 (0.0138) | 0.193 | about 0.12 |
| GoHome | 0.0521 (0.0253) | 1.823 | about 0.03 |

**Interpretation.**
- In absolute terms, `GoHome` deviates about 2.3 times more from its mean trajectory than `GoDown`. The 3D plot shows `GoHome` trials differing in extent.
- Relative to the size of the movement, the picture reverses: `GoDown` is a very small movement (about 3.5 cm of vertical range, judging from the axis of the plot, and a path length of 0.19), so a position offset of a couple of centimetres between trials is large compared with the gesture. In the plot, the `GoDown` trials look like similar arcs shifted sideways from trial to trial.
- So whether `GoHome` or `GoDown` is "more variable" depends on whether variability is measured in absolute or relative terms. Note also that the two 3D panels autoscale their axes and are therefore not drawn at the same scale.

**Limitations.** Five trials, two gestures, and the deviation includes position offsets between trials. A shape-only comparison would first remove position and orientation differences (for example with the alignment used in Section 3).

## 6. PCA: raw coordinates and engineered features

Principal Component Analysis (PCA) was applied separately to (i) the raw trajectory, resampled to 50 time points for each of 7 channels (350 values per recording), and (ii) the four engineered features (`path_length`, `total_rotation`, `straightness`, `net_rotation`). Duration and mean speed were left out because they change in the Faster and Slower contexts. Both representations were standardised first.

In [ ]:
N_POINTS = 50


def resample_raw(arr, n=N_POINTS):
    t = arr[:, 0]
    tn = (t - t[0]) / (t[-1] - t[0] + 1e-12)       # normalised time, so speed differences are removed
    grid = np.linspace(0, 1, n)
    return np.concatenate([np.interp(grid, tn, arr[:, j]) for j in range(1, 8)])


X_raw = np.array([resample_raw(r["data"]) for r in records])
FEATURE_COLS = ["path_length", "total_rotation", "straightness", "net_rotation"]
X_inv = table[FEATURE_COLS].to_numpy()
y = table["gesture"].to_numpy()

assert X_raw.shape[0] == X_inv.shape[0] == len(y)
assert np.isfinite(X_raw).all() and np.isfinite(X_inv).all()
print("Raw shape:", X_raw.shape, "| feature shape:", X_inv.shape)

gesture_names = sorted(set(y))
colors = plt.cm.tab10(np.arange(len(gesture_names)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, X, name in zip(axes, [X_raw, X_inv], ["Raw coordinates", "Invariant features"]):
    pca = PCA(n_components=2)
    Z = pca.fit_transform(StandardScaler().fit_transform(X))
    print(f"{name}: PC1 {pca.explained_variance_ratio_[0]:.4f}, PC2 {pca.explained_variance_ratio_[1]:.4f}, "
          f"total {100 * pca.explained_variance_ratio_.sum():.2f}%")
    for g, c in zip(gesture_names, colors):
        m = y == g
        ax.scatter(Z[m, 0], Z[m, 1], s=12, color=c, label=g, alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
axes[1].legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

**Result.**

| Representation | PC1 | PC2 | Combined |
|---|---:|---:|---:|
| Raw coordinates (350 values) | 28.34% | 19.45% | 47.79% |
| Engineered features (4 values) | 58.60% | 22.53% | 81.13% |

In the raw PCA, many tight clusters appear, but most clusters contain several gesture colours. In the engineered-feature PCA, some gestures form recognisable groups (for example `GoDown` towards the top and `GoHome` towards the lower left), while others overlap substantially.

**Interpretation.** The four engineered features give a much more compact summary: two components capture 81% of their variance against 48% for the raw representation. This reflects the different number of dimensions (4 against 350) and says nothing on its own about how well gestures can be told apart. The plots are coloured by gesture only, so I did not test whether the raw clusters correspond to contexts. PCA is unsupervised, so classification must be evaluated separately.

## 7. Classification under context change

**Question:** if a classifier is trained on one context, does it still recognise gestures in the other contexts, and does that depend on using raw coordinates or the simple features?

**Design.**
- The 420 recordings are variations of only 35 original recordings. A random split would put transformed copies of the same recording in both training and test sets and inflate accuracy.
- Instead, trials 1 to 3 of one context are used for training and trials 4 and 5, in every context, for testing (14 test samples per context, 21 training samples).
- Training is done once on `Original_1` and once on `Original_2`.
- The model is a standardised logistic regression; the scaler is fitted on training data only. Chance level for 7 balanced classes is 0.143.

**Assumption.** Trial *k* in every context comes from the same source recording *k*. I verified this for `Original_2` and `displaced_world_1` (Section 3) but assumed it for the other contexts.

In [ ]:
train_trials = [1, 2, 3]
test_trials = [4, 5]


def evaluate_representation(X, name, train_context):
    train_mask = ((table["context"] == train_context) & (table["trial"].isin(train_trials))).to_numpy()
    if train_mask.sum() == 0:
        raise ValueError(f"No training rows for {train_context}; check trial IDs.")
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    model.fit(X[train_mask], y[train_mask])
    rows = []
    for context in sorted(table["context"].unique()):
        test_mask = ((table["context"] == context) & (table["trial"].isin(test_trials))).to_numpy()
        pred = model.predict(X[test_mask])
        rows.append({"context": context, name: accuracy_score(y[test_mask], pred), "n_test": int(test_mask.sum())})
    return pd.DataFrame(rows).set_index("context")


accuracy = {}
for train_context in ["Original_1", "Original_2"]:
    raw_scores = evaluate_representation(X_raw, "raw", train_context)
    inv_scores = evaluate_representation(X_inv, "invariant", train_context)
    acc = raw_scores.join(inv_scores[["invariant"]])
    accuracy[train_context] = acc
    print(f"\nTraining context: {train_context}")
    display(acc.round(3))
    print("Mean context accuracy:", acc[["raw", "invariant"]].mean().round(3).to_dict())

print("\nChance-level reference for 7 balanced gesture classes:", round(1 / 7, 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (name, a) in zip(axes, accuracy.items()):
    a[["raw", "invariant"]].plot(kind="bar", ax=ax)
    ax.axhline(1 / 7, color="grey", linestyle="--", linewidth=1)
    ax.set_title(f"Trained on {name}")
    ax.set_ylabel("Held-out accuracy")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Results** (accuracy on held-out recordings, 14 test samples per context; the dashed line in the figure is chance, 0.143).

| Context | Raw (train `Original_1`) | Features (train `Original_1`) | Raw (train `Original_2`) | Features (train `Original_2`) |
|---|---:|---:|---:|---:|
| Combination | 0.14 | 0.00 | 0.14 | 0.00 |
| Faster | 1.00 | 0.36 | 1.00 | 0.50 |
| First_Half | 0.57 | 0.00 | 0.57 | 0.00 |
| Original_1 | 1.00 | 0.57 | 1.00 | 0.43 |
| Original_2 | 1.00 | 0.29 | 1.00 | 0.64 |
| Slower | 1.00 | 0.14 | 1.00 | 0.21 |
| displaced_body_1 | 0.14 | 0.21 | 0.14 | 0.50 |
| displaced_body_2 | 0.71 | 0.29 | 0.71 | 0.57 |
| displaced_body_3 | 0.57 | 0.21 | 0.57 | 0.50 |
| displaced_world_1 | 0.14 | 0.29 | 0.14 | 0.64 |
| displaced_world_2 | 0.14 | 0.21 | 0.14 | 0.57 |
| displaced_world_3 | 0.14 | 0.21 | 0.14 | 0.64 |
| **Mean over contexts** | **0.55** | **0.23** | **0.55** | **0.43** |

**Interpretation.**
1. **Raw coordinates fail when the frame changes.** They are perfect (1.00) in `Original_1`, `Original_2`, `Faster` and `Slower` (same frame; time-normalised resampling removes speed differences), but at chance (0.14) in all three world-frame contexts and in `displaced_body_1`. Raw accuracy differs across the three body-frame contexts (0.14, 0.71, 0.57); I did not investigate why.
2. **The simple features help only when trained on `Original_2`.** They then reach 0.57 to 0.64 in the world-frame contexts and 0.50 to 0.57 in the body-frame contexts, well above raw coordinates there. Trained on `Original_1`, they stay at 0.21 to 0.29.
3. **The difference between the two training contexts is probably related to Section 4c.** `Original_2` and the displaced contexts share a similar feature scale (path length and total rotation sums of about 1.2 and 7.7), whereas `Original_1` does not (0.96 and 4.0). A classifier trained on one scale is poorly calibrated for the other. This is a plausible explanation, not a tested one.
4. **The features are worse than raw coordinates when the frame is unchanged** (for example 0.21 against 1.00 in `Slower`), because the summed features depend on sampling and noise.
5. **Both representations fail on truncated trajectories.** The features reach 0.00 in `First_Half` and `Combination`, since a half trajectory has different sums; raw coordinates reach 0.57 in `First_Half` and 0.14 in `Combination`.
6. **The mean over contexts hides the pattern.** Raw coordinates look better overall (0.55 against 0.43), but only because most contexts keep the frame. The result that matters for reference-frame robustness is the world-frame rows.

**Limitations.** Each context has only 14 test samples, so one sample changes accuracy by 0.07; for example, 0.57 against 0.64 is a one-sample difference. The three world-frame contexts are built from the same 14 test recordings, so they are not independent replications. There is one train and test split, a small training set (21 samples), no tuning and no statistical test.

## 8. Conclusions, limitations and next steps

### What the analysis shows

- **Reference frames.** A world-frame change in this dataset is, to within a small residual, a rigid rotation and translation of the trajectory (35 of 35 pairs; residual about 0.0014 against 1.36 before alignment). Raw coordinates therefore differ greatly between frames although the movement geometry is preserved.
- **Features.** Path length and straightness are very stable under world-frame changes, but not under body-frame changes, where the tracked point moves. Total rotation is stable in both.
- **Summed features and sampling.** Path length and total rotation depend on how many samples a recording has and on noise, so they are weak invariants across contexts such as `Slower`, `Faster` and `Original_1` against `Original_2`.
- **Recognition across contexts.** Raw coordinates do not survive a frame change (chance level), whereas simple geometric features trained on a matching context recover a large part of the performance (0.57 to 0.64). Neither representation handles incomplete gestures.
- **Variability.** Trial-to-trial consistency depends on the gesture and on the feature, and conclusions about which gesture is more variable depend on whether absolute or relative variability is used.

### Limitations

- Only 7 gestures and 5 original recordings per gesture; all other contexts are derived from the same recordings, so they are not independent samples.
- Small test sets (14 per context) and a single train/test split; no statistical tests or confidence intervals.
- The trial correspondence across contexts was verified only for `Original_2` and `displaced_world_1`.
- The rigid-alignment check covers position only, for one context pair.
- My explanation of the sampling and noise effects is supported by several observations but was not tested directly.
- The features are simple hand-made summaries, not the frame-invariant measures a careful method would use, and some conclusions depend on assumptions about units (metres).
- This is a gesture-recognition dataset; nothing here concerns motor learning.

### Possible next steps

1. Smooth or denoise the trajectories before computing features, and test whether the differences across sampling rates and between `Original_1` and `Original_2` shrink.
2. Use features that are better frame-invariant or less dependent on sampling (for example time-normalised or shape-based descriptors) and check them on the body-frame and truncated contexts.
3. Verify trial correspondence and the rigid-alignment check across all context pairs, including orientation.
4. Use cross-validation over source recordings and report uncertainty.